All required import

In [49]:
import os
import sys
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer # used for embeddings

import warnings
warnings.filterwarnings("ignore")

In [50]:
# Load FAISS semantic corpus
df = pd.read_csv("../data/faiss_corpus.csv")

In [ ]:
# df['faiss_row'] = df.index # create an index so we can use it in LightGBM

In [51]:
print("\nSample faiss_text strings:")
for i in range(min(5, len(df))):
    print(f"  [{i}] {df['faiss_text'].iloc[i]}")


Sample faiss_text strings:
  [0] maze specialty coffee al barsha south 1 desserts coffee cafe low
  [1] loca jumeirah bar mexican mexican high
  [2] baofriend dubai silicon oasis (dso) arabic breakfast asian low
  [3] mayabay umm suqeim asian chinese japanese high
  [4] doner deli al garhoud fastfood sandwiches arabic medium


In [52]:
df = df.drop(columns=["venue_id"]) # we dont care about the venue we only want the text and the index
df.head()

,faiss_text
0,maze specialty coffee al barsha south 1 desser...
1,loca jumeirah bar mexican mexican high
2,baofriend dubai silicon oasis (dso) arabic bre...
3,mayabay umm suqeim asian chinese japanese high
4,doner deli al garhoud fastfood sandwiches arab...


SentenceTransformer doc: https://sbert.net/docs/sentence_transformer/pretrained_models.html

Model of choice doc: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

In [53]:
model = SentenceTransformer("all-MiniLM-L6-v2") #initialise our model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [54]:
sentences = df["faiss_text"].tolist() 

In [55]:
sentence_embedding = model.encode(
    sentences, 
    convert_to_numpy=True, # convert to numpy arrays
    normalize_embeddings=True  # behaves like cosine similiarity
    ).astype(np.float32) # prevent datatype incompatibility issues

In [56]:
print(f"Embedding matrix shape: {sentence_embedding.shape}")

Embedding matrix shape: (11603, 384)


Building our first FAISS index

In [ ]:
# dimentionality
d = sentence_embedding.shape[1]
d

384

In [57]:
index = faiss.IndexFlatIP(d) # initilize the index

In [59]:
# add our vectors
index.add(sentence_embedding)

In [60]:
index.ntotal # validate (number of embeddings in our index)

11603

In [ ]:
query = 'cheap arabic restaurant'

In [85]:
# start querying - query vector
xq = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

In [86]:
k = 10 # number of vectors to retrieve

In [87]:
%%time
# start the search
scores, indices = index.search(xq, k)
print(indices)

[[5196 1469 5149 3266 2302 5174 3648 4002 1382 2115]]
CPU times: user 1.51 ms, sys: 695 μs, total: 2.2 ms
Wall time: 1.12 ms


In [88]:
[f'{i}: {sentences[i]}' for i in indices[0]]

['5196: the daily restaurant arabic medium',
 '1469: the market restaurant & cafe international arabic medium',
 '5149: local house restaurant mediterranean arabic medium',
 '3266: apple cafe restaurant arabic arabic medium',
 '2302: alvand restaurant arabic arabic medium',
 '5174: arabian tea house restaurant arabic arabic medium',
 '3648: picnic restaurant arabic medium',
 '4002: pasargad restaurant arabic arabic medium',
 '1382: barjeel al arab restaurant arabic medium',
 '2115: bahar cuisine restaurant arabic arabic medium']

In [89]:
results = df.iloc[indices[0]].copy()
results["similarity_score"] = scores[0]
results[["faiss_text", "similarity_score"]]

,faiss_text,similarity_score
5196,the daily restaurant arabic medium,0.790948
1469,the market restaurant & cafe international ara...,0.788107
5149,local house restaurant mediterranean arabic me...,0.785613
3266,apple cafe restaurant arabic arabic medium,0.785591
2302,alvand restaurant arabic arabic medium,0.773118
5174,arabian tea house restaurant arabic arabic medium,0.772352
3648,picnic restaurant arabic medium,0.761999
4002,pasargad restaurant arabic arabic medium,0.761584
1382,barjeel al arab restaurant arabic medium,0.758333
2115,bahar cuisine restaurant arabic arabic medium,0.753530
